# M4 — Nurse rostering: the feasibility cliff

**2026 Quantum Accelerator · Day 1 · 75 minutes**

This morning you turned a knapsack into a matrix and learned that a penalty weight sits between
your constraints and your objective. Now we do it on a problem you already understand, and we
find out what happens to that trick when the rules pile up.

**The question for this module:** a roster problem has a fixed number of variables. If we keep the
variables fixed and only add *rules*, does the problem get harder — and if so, harder for whom?

**What you will do.** You will switch constraint families on one at a time and watch the same
solver, on the same variables, go from *always* finding a valid roster to *almost never* finding
one. Then you will claw some of it back two different ways, and finally compare against a mature
classical constraint solver.

**What you will type.** Four numbers, in four cells. Everything else is built for you.

---
## 0. The ward

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyqubo import Array, Constraint, Placeholder
from dwave.samplers import SimulatedAnnealingSampler
import helpers

# ------------------------------------------------------------------ the ward
N_NURSES = 12
N_DAYS   = 14
WEEKEND  = [5, 6, 12, 13]                 # two Sat/Sun pairs in a fortnight
SENIORS  = [0, 1, 2, 3]                   # four senior nurses

# how many nurses must be on duty each day
REQUIRED = np.array([3 if d in WEEKEND else 6 for d in range(N_DAYS)])
# ... of whom this many must be senior
SENIOR_REQUIRED = np.array([1 if d in WEEKEND else 2 for d in range(N_DAYS)])

DAYS_PER_NURSE    = int(REQUIRED.sum() // N_NURSES)            # 6 of 14
WEEKENDS_PER_NURSE = int(REQUIRED[WEEKEND].sum() // N_NURSES)  # 1 of 4

print(f"{N_NURSES} nurses x {N_DAYS} days = {N_NURSES * N_DAYS} binary decision variables")
print(f"on duty per day      : {REQUIRED.tolist()}")
print(f"of whom senior       : {SENIOR_REQUIRED.tolist()}")
print(f"days per nurse       : {DAYS_PER_NURSE}")
print(f"weekend days per nurse: {WEEKENDS_PER_NURSE}")
print()
print(f"search space         : 2^{N_NURSES * N_DAYS} = {2.0 ** (N_NURSES * N_DAYS):.3g} rosters")

168 variables is small. The search space is not: about 3.7 x 10^50 possible rosters, which is
more than the number of atoms in the Earth. Nothing in this module is solved by enumeration.

Each nurse also has preferences — days they would rather not work. This is our **objective**:
minimise the number of preference violations. The constraints decide what is *legal*; the
objective decides what is *good*.

In [ ]:
rng = np.random.default_rng(42)
PREFER_NOT = (rng.random((N_NURSES, N_DAYS)) < 0.30).astype(int)

print(f"{PREFER_NOT.sum()} 'would rather not' markers across the ward")
helpers.show_preferences(PREFER_NOT)

---
## 1. The model

Six families of rules. Read them once — you will not have to write them.

| # | family | what it says | how many constraints |
|---|---|---|---|
| 1 | `coverage` | the right number of nurses on duty each day | 14 |
| 2 | `skill_mix` | of whom the right number are senior | 14 |
| 3 | `weekend_fairness` | every nurse works exactly one weekend day | 12 |
| 4 | `workload` | every nurse works exactly six days | 12 |
| 5 | `rest_seniors` | no senior works two days in a row | 52 |
| 6 | `rest_all` | *nobody* works two days in a row | 156 |

Families 5 and 6 are the same rule applied to a widening group — a senior-staff agreement first,
then the whole ward. That is deliberate: it lets you see the collapse happen in two steps rather
than one.

Every family is wrapped in a PyQUBO `Constraint` with a **name**, and carries its own
`Placeholder` weight. The names are what let the solver tell us *which* rule it broke; the
placeholders are what let us compile the model once and change the weights afterwards.

In [ ]:
FAMILIES = ["coverage", "skill_mix", "weekend_fairness",
            "workload", "rest_seniors", "rest_all"]


def build_model(active, with_objective=True):
    """Build the roster QUBO using only the constraint families listed in `active`."""
    x = Array.create("x", shape=(N_NURSES, N_DAYS), vartype="BINARY")
    w = {f: Placeholder(f"w_{f}") for f in FAMILIES}      # one dial per family
    H = 0

    # ---- objective: every shift a nurse would rather not work costs 1 ----
    if with_objective:
        H += sum(int(PREFER_NOT[n, d]) * x[n, d]
                 for n in range(N_NURSES) for d in range(N_DAYS))

    # ---- 1. the right number of nurses on duty each day ----
    if "coverage" in active:
        for d in range(N_DAYS):
            H += w["coverage"] * Constraint(
                (sum(x[n, d] for n in range(N_NURSES)) - int(REQUIRED[d])) ** 2,
                label=f"coverage_day{d:02d}")

    # ---- 2. of whom the right number are senior ----
    if "skill_mix" in active:
        for d in range(N_DAYS):
            H += w["skill_mix"] * Constraint(
                (sum(x[n, d] for n in SENIORS) - int(SENIOR_REQUIRED[d])) ** 2,
                label=f"skillmix_day{d:02d}")

    # ---- 3. one weekend day each ----
    if "weekend_fairness" in active:
        for n in range(N_NURSES):
            H += w["weekend_fairness"] * Constraint(
                (sum(x[n, d] for d in WEEKEND) - WEEKENDS_PER_NURSE) ** 2,
                label=f"weekend_nurse{n:02d}")

    # ---- 4. six days each ----
    if "workload" in active:
        for n in range(N_NURSES):
            H += w["workload"] * Constraint(
                (sum(x[n, d] for d in range(N_DAYS)) - DAYS_PER_NURSE) ** 2,
                label=f"workload_nurse{n:02d}")

    # ---- 5 & 6. no two consecutive working days ----
    for fam, group in [("rest_seniors", SENIORS),
                       ("rest_all", [n for n in range(N_NURSES) if n not in SENIORS])]:
        if fam in active:
            for n in group:
                for d in range(N_DAYS - 1):
                    H += w[fam] * Constraint(x[n, d] * x[n, d + 1],
                                             label=f"rest_nurse{n:02d}_day{d:02d}")

    return H.compile()

Two helpers and we are ready. `weights()` fills in a dial setting for every family;
`anneal()` runs simulated annealing and reports how many of the runs produced a
**fully legal** roster.

In [ ]:
SAMPLER = SimulatedAnnealingSampler()


def weights(default=2.0, **overrides):
    """Penalty weight for every family, with named exceptions."""
    w = {f"w_{f}": float(default) for f in FAMILIES}
    w.update({f"w_{k}": float(v) for k, v in overrides.items()})
    return w


def anneal(model, w, num_reads=100, num_sweeps=2000, seed=1):
    """Run SA and summarise. `feasible_pct` = runs that broke no constraint at all."""
    bqm = model.to_bqm(feed_dict=w)
    sampleset = SAMPLER.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps, seed=seed)
    decoded = model.decode_sampleset(sampleset, feed_dict=w)

    broken = np.array([len(d.constraints(only_broken=True)) for d in decoded])
    legal = [d for d, b in zip(decoded, broken) if b == 0]
    best = min(legal, key=lambda d: d.energy) if legal else None

    return dict(feasible_pct=100 * len(legal) / len(decoded),
                mean_broken=float(broken.mean()),
                fewest_broken=int(broken.min()),
                n_constraints=len(decoded[0].constraints()),
                n_variables=len(bqm.variables),
                best=best)


def to_grid(decoded):
    """A decoded sample back into an N x D roster of 0/1."""
    return np.array([[decoded.array("x", (n, d)) for d in range(N_DAYS)]
                     for n in range(N_NURSES)])

---
## 2. Exercise 1 — one rule at a time

**You type one number.** Set `HOW_MANY` to `1`, run the cell, and read the output. Then set it to
`2`, run again. Keep going up to `4`.

Watch two things: the roster that comes back, and the `feasible` percentage.

In [ ]:
HOW_MANY = 1          # <-- CHANGE ME: 1, then 2, then 3, then 4

active = FAMILIES[:HOW_MANY]
model = build_model(active)
result = anneal(model, weights(default=2.0))

print(f"active families : {active}")
print(f"variables       : {result['n_variables']}")
print(f"constraints     : {result['n_constraints']}")
print(f"feasible runs   : {result['feasible_pct']:.0f}% of 100")
print()
if result["best"] is not None:
    helpers.render(to_grid(result["best"]), REQUIRED, SENIORS,
                   title="  a legal roster ('#' = on duty)")
else:
    print("  no legal roster found in 100 runs")

Up to four families the solver barely breaks a sweat. If you stopped here you would conclude that
annealing a roster is easy.

Now add the fifth family — and then the sixth.

---
## 3. Exercise 2 — the ladder

This cell walks all six rungs and records what happens. It takes a few seconds.

In [ ]:
labels, feas, broken, n_cons = [], [], [], []

for k in range(1, len(FAMILIES) + 1):
    active = FAMILIES[:k]
    r = anneal(build_model(active), weights(default=2.0))
    labels.append(FAMILIES[k - 1])
    feas.append(r["feasible_pct"])
    broken.append(r["mean_broken"])
    n_cons.append(r["n_constraints"])
    flag = "   <-- expected" if r["feasible_pct"] < 25 else ""
    print(f"{k}. +{FAMILIES[k-1]:<18} constraints={r['n_constraints']:>4}   "
          f"feasible={r['feasible_pct']:>5.1f}%   mean broken={r['mean_broken']:.2f}{flag}")

In [ ]:
helpers.plot_cliff(labels, feas, broken)

### Stop and look at that

The number of variables never changed. The solver never changed. The annealing budget never
changed. All that changed is how many rules we asked the objective function to carry — and the
solver went from finding a legal roster every single time to almost never.

This is the **encoding tax**. A QUBO has no notion of a constraint: we faked constraints by adding
a cost for breaking them. That works when there is plenty of room, and it degrades as the legal
region gets thin, because the solver spends its time in a landscape that is almost entirely
illegal.

**Talk to your table for two minutes.** In your own organisation, is the hard part usually
*optimising* something, or *finding anything that satisfies everybody*? Most rostering,
timetabling and planning problems are the second kind.

---
## 4. Exercise 3 — does more effort help?

The obvious reaction is "anneal harder". Let us find out what that buys.

**You type one number.** Add a bigger value to `SWEEP_BUDGETS` if you want to push further — but
mind the clock, the cost is roughly linear.

In [ ]:
SWEEP_BUDGETS = [500, 2000, 8000]      # <-- CHANGE ME: try adding 20000

stages = {"5 families (rest for seniors)": FAMILIES[:5],
          "6 families (rest for everyone)": FAMILIES[:6]}

series = []
for name, active in stages.items():
    m = build_model(active)
    row = [anneal(m, weights(default=2.0), num_sweeps=s)["feasible_pct"]
           for s in SWEEP_BUDGETS]
    series.append(row)
    print(f"{name:<32} " + "  ".join(f"{s}:{v:5.1f}%" for s, v in zip(SWEEP_BUDGETS, row)))

helpers.plot_effort(SWEEP_BUDGETS, series, list(stages))

Effort helps — and it does not rescue you. Sixteen times the annealing budget buys back some
feasibility on the full model, but you are still failing most of the time. Throwing compute at a
badly-conditioned landscape is not the lever.

So what is?

---
## 5. Exercise 4 — penalty balance

Every family has had the same weight so far: `default=2.0`. But the rest rule is structurally
different from the others. Coverage and workload are *quadratic deviation* penalties — breaking
one by a single nurse costs the same as one violation. The rest rule is a *single product* per
forbidden pair. There is no reason one dial should suit both.

**You type one list.** Sweep the weight on the rest rule while holding everything else at 2.0.

In [ ]:
REST_WEIGHTS = [0.5, 1, 2, 5, 10, 20]     # <-- CHANGE ME

model6 = build_model(FAMILIES)            # compiled ONCE - the placeholders do the rest
scores = []
for lam in REST_WEIGHTS:
    w = weights(default=2.0, rest_seniors=lam, rest_all=lam)
    r = anneal(model6, w, num_reads=100, num_sweeps=8000)
    scores.append(r["feasible_pct"])
    print(f"  weight on rest rule = {lam:>5}   feasible = {r['feasible_pct']:>5.1f}%   "
          f"fewest broken = {r['fewest_broken']}")

helpers.plot_lambda_balance(REST_WEIGHTS, scores)

### The real lesson about penalty weights

There is an interior optimum. Too low and the solver happily breaks the rest rule to satisfy
coverage. Too high and the rest rule flattens everything else into noise — the solver produces
beautifully rested nurses and the wrong number of them on duty.

Note what we did *not* do: we never recompiled the model. `Placeholder` let us build the QUBO once
and turn the dials afterwards. That is what makes this kind of exploration practical at all.

And note what this tells you about the morning's knapsack. λ was not a nuisance parameter you
tune once. With six families you have six dials, and it is their **balance** that matters, not
their size. This is the single most transferable skill in the field — and it is what you will be
scored on this afternoon.

---
## 6. The honest baseline

We have been treating "hard" as a property of the roster problem. It is not. It is a property of
*our encoding of it*.

Here is the identical model — same ward, same six families, same objective — given to OR-Tools
CP-SAT, a mature classical constraint solver. CP-SAT never flattens a constraint into a cost. It
knows what a constraint *is*.

In [ ]:
from ortools.sat.python import cp_model

def solve_with_cpsat(active, time_limit=20.0):
    m = cp_model.CpModel()
    x = {(n, d): m.NewBoolVar(f"x{n}_{d}")
         for n in range(N_NURSES) for d in range(N_DAYS)}

    if "coverage" in active:
        for d in range(N_DAYS):
            m.Add(sum(x[n, d] for n in range(N_NURSES)) == int(REQUIRED[d]))
    if "skill_mix" in active:
        for d in range(N_DAYS):
            m.Add(sum(x[n, d] for n in SENIORS) == int(SENIOR_REQUIRED[d]))
    if "weekend_fairness" in active:
        for n in range(N_NURSES):
            m.Add(sum(x[n, d] for d in WEEKEND) == WEEKENDS_PER_NURSE)
    if "workload" in active:
        for n in range(N_NURSES):
            m.Add(sum(x[n, d] for d in range(N_DAYS)) == DAYS_PER_NURSE)

    rest = set()
    if "rest_seniors" in active: rest |= set(SENIORS)
    if "rest_all" in active:     rest |= set(range(N_NURSES))
    for n in sorted(rest):
        for d in range(N_DAYS - 1):
            m.Add(x[n, d] + x[n, d + 1] <= 1)        # a constraint, not a cost

    m.Minimize(sum(int(PREFER_NOT[n, d]) * x[n, d]
                   for n in range(N_NURSES) for d in range(N_DAYS)))

    s = cp_model.CpSolver()
    s.parameters.max_time_in_seconds = float(time_limit)
    s.parameters.num_workers = 4
    status = s.Solve(m)
    grid = None
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        grid = np.array([[s.Value(x[n, d]) for d in range(N_DAYS)]
                         for n in range(N_NURSES)])
    return s.StatusName(status), s.WallTime(), grid, s.ObjectiveValue()


status, wall, grid, obj = solve_with_cpsat(FAMILIES)
print(f"CP-SAT: {status} in {wall:.3f} s, preference violations = {obj:.0f}")
print()
helpers.render(grid, REQUIRED, SENIORS, title="  CP-SAT optimal roster")

In [ ]:
# side by side, on the full six-family model, both with their best settings
import time

t0 = time.perf_counter()
sa = anneal(model6, weights(default=2.0, rest_seniors=5, rest_all=5),
            num_reads=200, num_sweeps=8000)
sa_wall = time.perf_counter() - t0
sa_cost = int((to_grid(sa["best"]) * PREFER_NOT).sum()) if sa["best"] else None

print(f"{'':<36}{'legal?':>9}{'pref. cost':>12}{'time':>10}")
print(f"{'simulated annealing (best of 200)':<36}"
      f"{str(sa['feasible_pct'] > 0):>9}{str(sa_cost):>12}{f'{sa_wall:.2f} s':>10}")
print(f"{'CP-SAT (proved optimal)':<36}{'True':>9}{int(obj):>12}{f'{wall:.3f} s':>10}")
print()
print(f"annealing found a legal roster in {sa['feasible_pct']:.0f}% of its runs, "
      f"and needed {sa_wall / wall:.0f}x longer to do it")

---
## 7. What just happened

1. **Variables are not the difficulty. Constraint density is.** 168 variables stayed 168 variables
   while the problem went from trivial to intractable *for this encoding*.

2. **A QUBO cannot represent a constraint.** It can only represent a cost for breaking one. That
   substitution is the encoding tax, and you pay it in solution quality exactly when the legal
   region gets thin.

3. **Penalty balance beats penalty magnitude, and beats extra compute.** Re-weighting one family
   bought more feasibility than a sixteen-fold increase in annealing effort.

4. **A mature constraint solver is still the thing to beat.** CP-SAT solved this in milliseconds
   and certified the answer. Be suspicious of anyone who shows you a quantum or quantum-inspired
   result without a classical baseline in the same notebook.

So why does anyone build QUBO solvers at all? Because some problems have no compact constraint
structure to exploit, some have millions of variables, and some — you will see one after lunch —
defeat CP-SAT too. That is the honest case, and it is the one we will make.

---
## 8. Stretch goals

If you finished early, pick one.

**A. Soft constraints.** Move `weekend_fairness` out of the constraint list and into the objective
with a small weight, so that an unfair weekend is *discouraged* rather than *forbidden*. Does the
feasibility cliff move?

**B. Per-family tuning.** You tuned one dial in Exercise 4. Tune two: `rest_*` and `coverage`. Can
you beat the best feasibility you saw? Record your best combination — it is worth knowing before
the tournament.

**C. Scale it.** Change `N_NURSES` to 24 and `N_DAYS` to 28 (672 variables). Re-run Exercise 2.
Does the cliff arrive at the same rung, earlier, or later?

**D. A different heuristic.** Swap `SimulatedAnnealingSampler` for `TabuSampler` from
`dwave.samplers` on the six-family model. Same encoding, different search. Does the encoding tax
depend on the searcher?

**E. Check the published formulation.** Ikeda, Nakamura & Humble (*Scientific Reports* 9:12837,
2019) published a nurse-scheduling QUBO with three components rather than six. Read their
formulation and work out which of our six families they left out — and what that tells you about
what fits comfortably in a QUBO.